In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================
# 1. Google Drive を接続
# =========================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================
# 2. ライブラリ読み込み
# =========================================
import pandas as pd
import requests
import json

# =========================================
# 3. 設定
# =========================================
CSV_PATH = "/content/drive/MyDrive/課題用/自分用プロトタイプ/input.csv"   # ← 自分のCSVパスに変更
OUTPUT_CSV_PATH = "/content/drive/MyDrive/課題用/自分用プロトタイプ/output.csv"  # 出力先
DIFY_API_KEY = "★自分のAPIキー★"
DIFY_WORKFLOW_URL = "https://api.dify.ai/v1/workflows/run"

# =========================================
# 4. CSV読み込み
#    utf-8-sig でBOM付きUTF-8にも対応
# =========================================
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

# 列名の前後空白を削除
df.columns = df.columns.str.strip()

print("列名:", df.columns.tolist())
print(df.head())

# =========================================
# 5. 最新行を取得
#    必須列が全部空の行は除外
# =========================================
required_cols = ["年齢", "身長_cm", "体重_kg", "性別", "活動レベル"]
clean_df = df.dropna(subset=required_cols, how="all")

if clean_df.empty:
    raise ValueError("CSVに有効なデータがありません。")

latest = clean_df.iloc[-1]

age = int(latest["年齢"])
height = float(latest["身長_cm"])
weight = float(latest["体重_kg"])
sex = str(latest["性別"]).strip()
activity = str(latest["活動レベル"]).strip()

print("最新行:", age, height, weight, sex, activity)

# =========================================
# 6. 入力値チェック
# =========================================
if sex not in ["男", "女"]:
    raise ValueError("性別は『男』または『女』で入力してください。")

activity_factor = {
    "低い": 1.50,
    "普通": 1.75,
    "高い": 2.00
}

if activity not in activity_factor:
    raise ValueError("活動レベルは『低い』『普通』『高い』で入力してください。")

# =========================================
# 7. 基礎代謝量(BMR)計算
#    ※この式は今回のサンプル実装
# =========================================
if sex == "男":
    bmr = round(10 * weight + 6.25 * height - 5 * age + 5)
else:
    bmr = round(10 * weight + 6.25 * height - 5 * age - 161)

# =========================================
# 8. 必要エネルギー量計算
# =========================================
energy = round(bmr * activity_factor[activity])

# =========================================
# 9. 食事バランスガイド用の目安SV計算
#    ※今回のサンプルルール
# =========================================
def calculate_sv(energy_kcal: int):
    if energy_kcal < 1800:
        return 4, 5, 3, 2, 1
    elif energy_kcal < 2200:
        return 5, 6, 4, 2, 2
    else:
        return 6, 7, 5, 2, 2

grain, veg, main, milk, fruit = calculate_sv(energy)

print("BMR:", bmr)
print("必要エネルギー量:", energy)
print("SV:", grain, veg, main, milk, fruit)

# =========================================
# 10. Dify Workflow へ送信
# =========================================
headers = {
    "Authorization": f"Bearer {DIFY_API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "inputs": {
        "age": int(age),
        "height_cm": float(height),
        "weight_kg": float(weight),
        "sex": str(sex),
        "activity_level": str(activity),
        "bmr": int(bmr),
        "energy_kcal": int(energy),
        "grain_sv": int(grain),
        "vegetable_sv": int(veg),
        "main_sv": int(main),
        "milk_sv": int(milk),
        "fruit_sv": int(fruit)
    },
    "response_mode": "blocking",
    "user": "colab-user-001"
}

response = requests.post(
    DIFY_WORKFLOW_URL,
    headers=headers,
    data=json.dumps(payload),
    timeout=120
)

print("status_code:", response.status_code)
print("raw_response:", response.text)

response.raise_for_status()
result = response.json()

# =========================================
# 11. Difyの出力を取得
#    Outputノードで result という出力変数名にしている前提
# =========================================
menu_text = result["data"]["outputs"]["result"]
print("Dify出力:", menu_text)

# =========================================
# 12. JSONを分解
# =========================================
menu = json.loads(menu_text)

breakfast = menu.get("breakfast", "")
lunch = menu.get("lunch", "")
dinner = menu.get("dinner", "")
comment = menu.get("comment", "")

print("朝食:", breakfast)
print("昼食:", lunch)
print("夕食:", dinner)
print("コメント:", comment)

# =========================================
# 13. 結果をCSV出力
# =========================================
out_df = pd.DataFrame([{
    "年齢": age,
    "身長_cm": height,
    "体重_kg": weight,
    "性別": sex,
    "活動レベル": activity,
    "基礎代謝量_kcal": bmr,
    "必要エネルギー量_kcal": energy,
    "主食_SV": grain,
    "副菜_SV": veg,
    "主菜_SV": main,
    "乳製品_SV": milk,
    "果物_SV": fruit,
    "朝食案": breakfast,
    "昼食案": lunch,
    "夕食案": dinner,
    "補足コメント": comment
}])

out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"出力完了: {OUTPUT_CSV_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
列名: ['年齢', '身長_cm', '体重_kg', '性別', '活動レベル']
   年齢  身長_cm  体重_kg 性別 活動レベル
0  37    159     55  女    普通
最新行: 37 159.0 55.0 女 普通
BMR: 1198
必要エネルギー量: 2096
SV: 5 6 4 2 2
status_code: 200
raw_response: {"task_id": "bb7ff1d8-c721-4dd5-92ee-5f20744f9459", "workflow_run_id": "55015743-59bd-4603-89e7-81f8ae919cc7", "data": {"id": "55015743-59bd-4603-89e7-81f8ae919cc7", "workflow_id": "9a54ffaf-d447-43f0-8ed6-3ff622e41967", "status": "succeeded", "outputs": {"result": "{\n  \"breakfast\": \"\u3054\u98ef\uff08\u8336\u78971\u676f\uff09\uff0b\u5869\u9bad\u306e\u713c\u304d\u7269\uff08\u5c0f1\u5207\u308c\uff09\uff0b\u307b\u3046\u308c\u3093\u8349\u306e\u304a\u6d78\u3057\uff0b\u5473\u564c\u6c41\uff08\u8c46\u8150\u3068\u308f\u304b\u3081\uff09\uff0b\u30e8\u30fc\u30b0\u30eb\u30c8\uff08\u7121\u7cd6\uff09\u3002\u7406\u7531: \u624b\u65e9\u304f\u4f5c\u308c\u3066\u4e3b\u98df\u30fb\u4e